In [59]:
import pandas as pd

df = pd.read_csv("healthcare-dataset-stroke-data.csv")

print("Shape:", df.shape)
print(df.head())
print(df.isnull().sum())

Shape: (5110, 12)
      id  gender   age  hypertension  heart_disease ever_married  \
0   9046    Male  67.0             0              1          Yes   
1  51676  Female  61.0             0              0          Yes   
2  31112    Male  80.0             0              1          Yes   
3  60182  Female  49.0             0              0          Yes   
4   1665  Female  79.0             1              0          Yes   

       work_type Residence_type  avg_glucose_level   bmi   smoking_status  \
0        Private          Urban             228.69  36.6  formerly smoked   
1  Self-employed          Rural             202.21   NaN     never smoked   
2        Private          Rural             105.92  32.5     never smoked   
3        Private          Urban             171.23  34.4           smokes   
4  Self-employed          Rural             174.12  24.0     never smoked   

   stroke  
0       1  
1       1  
2       1  
3       1  
4       1  
id                     0
gender       

In [60]:
# ลบ id เพราะไม่มีความหมายเชิงพยากรณ์
df.drop("id", axis=1, inplace=True)

In [61]:
print(df.isnull().sum())

gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


In [62]:
from sklearn.impute import SimpleImputer

bmi_imputer = SimpleImputer(strategy="median")
df["bmi"] = bmi_imputer.fit_transform(df[["bmi"]])

In [63]:
X = df.drop("stroke", axis=1)
y = df["stroke"]

In [64]:
X = pd.get_dummies(X, drop_first=True)

In [65]:
print(y.value_counts())

stroke
0    4861
1     249
Name: count, dtype: int64


In [66]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [67]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [68]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4088, 16)
Test shape: (1022, 16)


In [69]:
print(y_train.value_counts())

stroke
0    3889
1     199
Name: count, dtype: int64


In [70]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# คำนวณ class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {
    0: class_weights[0],
    1: class_weights[1]
}

print("Class Weights:", class_weight_dict)

# สร้างโมเดล
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(16, activation='relu'),

    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

Class Weights: {0: np.float64(0.5255849832861919), 1: np.float64(10.271356783919598)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.5439 - loss: 0.7775 - val_accuracy: 0.6235 - val_loss: 0.6629
Epoch 2/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6260 - loss: 0.6074 - val_accuracy: 0.6675 - val_loss: 0.6013
Epoch 3/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6632 - loss: 0.5596 - val_accuracy: 0.6785 - val_loss: 0.5660
Epoch 4/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6773 - loss: 0.5523 - val_accuracy: 0.7200 - val_loss: 0.5229
Epoch 5/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6955 - loss: 0.5444 - val_accuracy: 0.7225 - val_loss: 0.5260
Epoch 6/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6937 - loss: 0.5216 - val_accuracy: 0.7518 - val_loss: 0.4802
Epoch 7/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7053 - loss: 0.5443 - val_accuracy: 0.7384 - val_loss: 0.4638
Epoch 8/100
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7049 - loss: 0.5042 - val_

In [71]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7051 - loss: 0.4492 
Test Accuracy: 0.6966732144355774


In [72]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_test) > 0.5).astype("int32")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Confusion Matrix:
[[672 300]
 [ 10  40]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.69      0.81       972
           1       0.12      0.80      0.21        50

    accuracy                           0.70      1022
   macro avg       0.55      0.75      0.51      1022
weighted avg       0.94      0.70      0.78      1022



In [73]:
y_prob = model.predict(X_test)

# ลอง threshold = 0.7
y_pred_custom = (y_prob > 0.7).astype("int32")

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
[[846 126]
 [ 19  31]]
              precision    recall  f1-score   support

           0       0.98      0.87      0.92       972
           1       0.20      0.62      0.30        50

    accuracy                           0.86      1022
   macro avg       0.59      0.75      0.61      1022
weighted avg       0.94      0.86      0.89      1022



In [74]:
model.save("stroke_nn_model.h5")

In [75]:
model.save("stroke_nn_model.keras")

In [76]:
import joblib
joblib.dump(scaler, "stroke_scaler.pkl")

['stroke_scaler.pkl']

In [77]:
joblib.dump(X.columns, "stroke_columns.pkl")

['stroke_columns.pkl']